# Práctica: Inyección de Contexto y "RAG Simplificado"

Ya vimos como hablar con el LLM, hoy vamos a darle "memoria" sobre nuestros propios datos. Esta técnica de inyectar nuestros propios documentos en la memoria de trabajo del modelo es uno de los aspectos más importantes en la aplicación práctica de los LLM.

Contenidos:
1. Leer un archivo de texto para incrustarlo como contexto.
2. Construir un "Super-Prompt" enlazando las instrucciones y el texto del documento.
3. Forzar a Gemini a no inventar (alucinar) respuestas si la información no está en el texto.
4. Obligar al modelo a devolver datos estructurados, como diccionarios JSON (imprescindible para APIs).

---

## 1. Configuración Inicial de la API

Lo primero de todo, instalemos la librería de nuevo y configuremos la API Key como hicimos ayer.

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=GOOGLE_API_KEY)

#Instanciamos el modelo. Usaremos gemini-3.1-flash-lite-preview. Si hay problemas de cuota/modelo, siempre es posible crear un nuevo proyecto, API Key y usar otro (con menos peticiones diarias) como gemini-2.5-flash
MODEL_ID = "models/gemini-3.1-flash-lite-preview"

---
## 2. Leer un Archivo .txt o .md en código
Primero necesitamos crear un archivo simulando la base de datos de nuestra empresa. Lo leeremos con código puro de Python.

In [ ]:
contenido_del_archivo = """
# Políticas Internas EmpresaFalsa123.

## Trabajo Remoto
- Los empleados pueden trabajar desde casa un máximo de 3 días a la semana.
- Es obligatorio estar online en Slack entre las 10:00 y las 14:00.

## Vacaciones 2026
- Todos tienen 25 días laborables de vacaciones pagadas.
- Durante el mes de Agosto no se pueden tomar más de 2 semanas consecutivas.

## Comidas y Oficina
- El menú de la cafetería cambia cada semana. El menú del viernes es siempre "Día de Pizza".
- La oficina cierra a las 20:00 y requiere tarjeta magnética para el acceso de fin de semana.
"""

with open("politicas_empresa.md", "w", encoding="utf-8") as f:
    f.write(contenido_del_archivo)

print("Archivo 'politicas_empresa.md' guardado en disco")

---
### Ejercicio 1: Inyección de Contexto en el Prompt

Ahora tienes este archivo en disco, pero el modelo Gemini en la nube no lo puede ver. Tienes que leer el archivo y mandárselo **dentro** del String de tu prompt. 

**Tu turno:** 
1. Abre y lee el contenido del archivo `politicas_empresa.md` y coge el string.
2. Crea una variable con una `pregunta_usuario` inventada sobre este texto.
3. Concatena todo usando *f-strings* siguiendo el patrón de teoría para aislar el contexto con `--- INICIO DE INFORMACION ---` e instruyendo a que no invente la respuesta.

In [ ]:
#EJERCICIO 1: "RAG Simplificado" con Petición Única.

#TODO: Lee tu archivo guardado y almacena su contenido en una variable (usa open...)

#TODO: Crea la pregunta

#TODO: Ensambla el prompt completo.
prompt_con_contexto = f"""
Eres un asistente experto de EmpresaFalsa123. 
Responde usando ESTRICTAMENTE la siguiente información. Si no lo sabes, no lo inventes.

--- INICIO DE LA INFORMACIÓN ---


--- FIN DE LA INFORMACIÓN ---

Pregunta del usuario: 
"""

print(prompt_con_contexto)

#TODO: Haz la petición a Gemini y comprueba si responde correctamente basándose en el texto.

---
## 3. Controlando el Formato de Salida (Extracción a JSON)

Las aplicaciones normalmente se deben comunicar con otros sistemas web. Devolver parrafadas como "¡Claro, aquí tienes la respuesta: 25 días!" es horrible para una API de frontend.  
A veces, no queremos que el asistente nos responda con texto humano, sino que queremos extraer datos estructurados del contexto en un Json estricto.

### Ejercicio 2: Extracción de Estructuras (JSON)
Vamos a crear otro script que lea nuestro archivo `politicas_empresa.md`. Pero esta vez el prompt que generéis no será para "Charlar" con él, sino para extraer datos y convertirlos de texto libre a un diccionario JSON limpio que programáticamente podamos usar en Python.

**Qué hay que hacer:**
Prepara las instrucciones (System Prompts) necesarias para que **obliguen** a Gemini a devolver el número máximo de días de teletrabajo a la semana y los días totales de vacaciones que tiene un empleado en 2026.
El output esperado del modelo debe ser algo así para que sea programable en Python:
```json
{
  "dias_max_teletrabajo": 3,
  "dias_vacaciones_2026": 25
}
```
*Además del prompt, recuerda usar Temperature=0.0 para extracciones certeras y que así no se invente nada.*

*Para obligar a que elija el formato json, también es útil añadir en la configuración `response_mime_type="application/json"`*

In [ ]:
#EJERCICIO 2: Formato de Salida

import json
from google.genai import types

#Extrae el texto si no lo tenías.
with open("politicas_empresa.md", "r", encoding="utf-8") as f:
    texto_politicas = f.read()

#TODO: Crea el prompt completo
#Dependiendo del formato de salida, lo más recomendable es indicar el esquema que debe seguir, utilizando ejemplos
prompt_extraccion = f"""
  ... (aquí tu System Prompt muy detallado de la tarea y de qué tiene que devolver OBLIGATORIAMENTE JSON)...

  
  -- TEXTO -- 
  {texto_politicas}
"""

#TODO: Crea un config de generación con temperature=0 y application/json

#TODO: Haz la petición a Gemini con el prompt de extracción y el config de generación.
response = ...

#Vamos a ver si ha funcionado y si el formato de salida es correcto (JSON)
texto_respuesta = response.text 
print("La respuesta ha sido:")
print(texto_respuesta)

#Vamos a probar si funciona el formato json de la respuesta. Si el prompt es correcto, debería devolver un JSON con la información solicitada y se podría usar la salida de Gemini directamente para enviársela a cualquier servicio
try:
    mi_diccionario = json.loads(texto_respuesta)
    print("La respuesta es un JSON válido.")
except json.JSONDecodeError:
    print("La respuesta no es un JSON válido.")

print(mi_diccionario['dias_vacaciones_2026'])

---
### Ejercicio 3: Extracción de Listas (JSON array)
Ahora vamos a pedirle al modelo que extraiga una lista de todas las reglas de la oficina y las devuelva en una estructura de lista (array de JSON).

**Qué hay que hacer:**
Prepara las instrucciones para que Gemini devuelva una lista (array en JSON) de todas las reglas presentes bajo la sección "Comidas y Oficina" y "Trabajo Remoto". Además de un número que indique los días de vacaciones.
El output esperado del modelo debe ser algo así:
```json
{
  "reglas": [
    "Los empleados pueden trabajar desde casa un máximo de 3 días a la semana",
    "Es obligatorio estar online en Slack entre las 10:00 y las 14:00",
    ...
  ],
  "dias_vacaciones": 22
}
```

El JSON schema se hace de la forma siguiente:
```python
schema = types.Schema(
    type="OBJECT", #Para indicarle que devuelva un json con campos { campo1: ..., campo2:... }
    properties={
        "nombre_campo": types.Schema( #Un Schema por cada campo que queramos en el JSON
            type="TIPO_DATO", #STRING, NUMBER, ARRAY...
            
            #Solo si type="ARRAY", hay que indicar de qué tipo queremos que sea cada elemento del array
            items=types.Schema(type="TIPO_DATO")
        )
    },
    required=["nombre_campo"] #Solo si los campos tienen que aparecer obligatoriamente
)
```

*Se puede indicar un esquema JSON en la configuración con response_schema=schema*


In [ ]:
#EJERCICIO 3: Lista de Reglas (JSON Array)

#TODO: Crear el prompt con las reglas que debe seguir la salida, indicando que debe ser un JSON Array con un formato específico
prompt_lista = f"""
  ...(Instrucciones para extraer una lista en JSON plano, sin backticks de markdown ni florituras o bien extrayendola programáticamente en python con json.loads) ...

  -- TEXTO --
  {texto_politicas}
"""

#TODO: Crea un esquema que indique que es un array que a su vez contiene strings

#TODO: Crea un config de generación con temperature=0, application/json y el esquema de salida que has creado

#TODO: Haz la petición a Gemini con el prompt de extracción y el config de generación.
response = ...

#Vamos a ver si ha funcionado y si el formato de salida es correcto (JSON)
texto_respuesta = response.text 
print("La respuesta ha sido:")
print(texto_respuesta)

lista_reglas = json.loads(texto_respuesta)
print(lista_reglas)

---
## 4. RAG + Chat

Vamos a unir las dos piezas: el **Contexto (RAG)** de hoy con el **Chat (Historial)** que vimos ayer en los conceptos básicos. 

En la API de Gemini, hay una forma muy potente de configurar esto: al instanciar el modelo Base, podemos pasarle un parámetro oculto llamado `system_instruction`. Si metemos ahí todo nuestro contexto (nuestros documentos y reglas), el modelo asumirá esa información como el "background" en el que se basará todo el modelo de chat.

### Ejercicio 4: Chatbot conversacional con tu archivo .md
Completa el script para crear un bucle de terminal (`input()`) donde le puedas pedir al chatbot cosas relacionadas con la empresa, y comprueba probando tú mismo si se acuerda de tu mensaje de la pregunta anterior (y de las políticas de empresa al mismo tiempo).

In [ ]:
#EJERCICIO 4: Bucle de Chat con Contexto (RAG + Historial)

#Leemos el documento local (simulando la Extracción/RAG)
with open("politicas_empresa.md", "r", encoding="utf-8") as f:
    documento = f.read()

#Definimos las reglas inquebrantables del chatbot y adjuntamos las políticas
instrucciones_sistema = f"""
Eres el asistente virtual de RRHH de EmpresaFalsa123. 
Responde de forma concisa y amigable basándote EXCLUSIVAMENTE en el siguiente documento.
Si el usuario te pregunta por algo (ej. salarios) que no aparezca en las políticas provistas, NO TE LO INVENTES,
responde educadamente que no tienes acceso a esa información.

--- INICIO DEL DOCUMENTO OFICIAL ---
{documento}
--- FIN DEL DOCUMENTO OFICIAL ---
"""

#TODO: Crea una configuración con el "Super-Prompt", a diferencia de un prompt de un solo uso

#TODO: Crea un modelo de chat con la configuración anterior, de manera que cada vez que crees un nuevo chat derivado de este modelo, el asistente ya sepa qué tiene que hacer por detrás sin decírselo de nuevo.

#TODO: Haz un bucle infinito que permita al usuario escribir preguntas en la terminal (usando input(...) ) y el chatbot responderlas, utilizando el modelo de chat que has creado. El bucle se romperá cuando el usuario escriba "salir", "exit" o "quit".

print("=== Chatbot RRHH EmpresaFalsa123 iniciado (Escribe 'salir' para terminar) ===")


#-> CUANDO TENGAS EL CÓDIGO:
#Ejecuta la celda.
#1º Pregúntale si puedes cogerte diciembre entero de vacaciones. (debería decir que no lo sabe, no está en el doc. Otra opción es que te diga que sí porque no se especifica en el doc)
#2º Pregúntale si puedes ir en Agosto entero (debería decirte que solo 2 semanas máximo según tu .md)
#3º Hazle una repregunta "Y entonces en navidades?" (debería acordarse de qué se refería "entonces" por el contexto del historial).

---
## 5. RAG con citas verificables

En proyectos reales no basta con responder: hay que **justificar** la respuesta con evidencia (como ya vimos en los RAGs en modelos de Hugging Face).  
En este ejercicio extenderás el RAG simplificado para que devuelva **respuesta + citas literales** del documento.

### Ejercicio 5: RAG con citas (respuesta con evidencia)

**Qué hay que hacer:**
1. Leer `politicas_empresa.md` como contexto.
2. Construir un prompt que obligue a responder en JSON con este formato:
```json
{
  "respuesta": "...",
  "citas": ["...", "..."],
  "hay_evidencia": true
}
```
3. Si no hay evidencia en el documento, debe devolver:
```json
{
  "respuesta": "No tengo información suficiente en el documento.",
  "citas": [],
  "hay_evidencia": false
}
```
4. Añadir una validación en Python que compruebe que cada cita aparece de forma literal en el documento.

Sugerencia: usa `temperature=0.0`, `response_mime_type="application/json"` y una instrucción explícita de "no inventar citas".

---
## 6. Fallback y degradación controlada

En sistemas reales habrá fallos de red, cuota, timeout o errores de proveedor.  
Un asistente útil debe degradar de forma controlada y no romper la experiencia.

### Ejercicio 6: Estrategia de fallback

**Qué hay que hacer:**
1. Implementar una función de consulta robusta con:
- timeout
- reintentos con backoff simple
- fallback a un segundo modelo si falla el principal
2. Si ambos fallan, devolver una respuesta de contingencia clara para usuario.
3. Registrar qué estrategia se usó (`principal`, `fallback`, `contingencia`).

El objetivo es que la aplicación **siempre** responda algo útil y observable, incluso cuando la IA falle.